In [1]:
import numpy as np
import pandas as pd
import pickle
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Load from Feature Engineering output
train = pd.read_csv('data_train.csv')
val   = pd.read_csv('data_val.csv')
with open('feature_cols.pkl', 'rb') as f:
    obj = pickle.load(f)
FEATURE_COLS  = obj['FEATURE_COLS']
global_median = obj['global_median']
print(f'Train: {len(train):,} rows | Val: {len(val):,} rows')
print(f'Features: {len(FEATURE_COLS)}')

Train: 301,355 rows | Val: 4,871 rows
Features: 27


# Modeling
We will compare two approaches according to the problem instructions:
- Tier 1: Global Model → one RandomForest for all products
- Tier 2: Product Level → use the last price per product

Both use anchor calibration for prediction correction.

## 1. Tier 1 → Global Model (RandomForest)
One model is trained on the entire training data. The model looks at all features and finds general patterns across the entire marketplace.

RandomForest is chosen because:
- No scaling required → tree-based models are not sensitive to the scale of numbers
- Robust to outliers → voting/averaging across many trees reduces the impact of outliers
- Interpretable → can view feature importance
- Suitable for tabular data → the data used in this project is tabular data

In [2]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

print("Training Tier 1 (Global Model)... wait ~1-2 minutes")

model = RandomForestRegressor(
    n_estimators=100,   # 100 trees
    max_depth=15,       # trees should not be too deep (prevent overfitting)
    min_samples_leaf=5, # minimum 5 samples per leaf
    random_state=42,    # ensures reproducible results across runs
    n_jobs=-1           # use all CPU cores
)

X_train = train[FEATURE_COLS]
y_train = train['log_price']   # target = log(price)

model.fit(X_train, y_train)
print("Done!")

Training Tier 1 (Global Model)... wait ~1-2 minutes
Done!


#### Tier 1 Evaluation (Without Calibration)
See how accurate the model is without anchor calibration.

In [3]:
X_val     = val[FEATURE_COLS]
pred_log  = model.predict(X_val)
pred_val  = np.expm1(pred_log)    # convert back from log to IDR
true_val  = val['price'].values

mae  = mean_absolute_error(true_val, pred_val)
mape = (abs(true_val - pred_val) / true_val).mean() * 100

print(f"── Tier 1 (Without Calibration) ──")
print(f"MAE  : Rp {mae:,.0f}")
print(f"MAPE : {mape:.2f}%")


── Tier 1 (Without Calibration) ──
MAE  : Rp 46,269
MAPE : 0.20%


MAE shows the average difference between the prediction and the actual price in Rupiah. MAPE shows the difference in percentage. MAPE is a fairer metric because it is not affected by the price scale. Being off by Rp 50k on a Rp 100k product has a completely different meaning than being off by Rp 50k on a Rp 10 million product.

#### Anchor Calibration
100 known prices on the validation day are used to detect whether there is a price shift on that day.

How it works: calculate the ratio of the actual anchor price / model prediction for the anchor. If ratio > 1 → prices on that day are higher than predicted. If ratio < 1 → prices on that day are lower than predicted. Then, correct all predictions using that ratio.


In [4]:
# Anchor simulation, 100 random rows from val
anchor      = val.sample(n=100, random_state=42)
to_predict  = val.drop(anchor.index)

# Model predictions for anchor
anchor_pred = np.expm1(model.predict(anchor[FEATURE_COLS]))
anchor_true = anchor['price'].values

# Ratio: actual price / model prediction
# Use median because it is more robust to outliers than the mean
rasio = np.median(anchor_true / anchor_pred)

print(f"Global ratio from 100 anchors: {rasio:.4f}")
print(f"Meaning the price on that day is {(rasio-1)*100:+.2f}% compared to the model prediction")

Global ratio from 100 anchors: 1.0000
Meaning the price on that day is -0.00% compared to the model prediction


If the ratio = 1.0000, it means there is no price shift on that day because the model is already accurate without needing correction. In real-world scenarios such as flash sales or platform-wide price hikes, the ratio can differ significantly, and calibration will be highly beneficial.

#### Tier 1 Evaluation (With Calibration)
See how accurate the model is with anchor calibration.

In [5]:
# Tier 1 with calibration
pred_all       = np.expm1(model.predict(to_predict[FEATURE_COLS]))
pred_calibrated = pred_all * rasio

mae_t1  = mean_absolute_error(to_predict['price'], pred_calibrated)
mape_t1 = (abs(to_predict['price'] - pred_calibrated) / to_predict['price']).mean() * 100

print(f"{'Tier 1 + anchor calibration':<35} Rp{mae_t1:>13,.0f} {mape_t1:>7.2f}%")

Tier 1 + anchor calibration         Rp       47,124    0.20%


#### Tier 2 Evaluation
Tier 2 uses the last price per product as the prediction, corrected by the ratio from the anchor.

In [6]:
# Tier 2: last price × ratio
tier2_pred = to_predict['model_last_price'] * rasio
mae_t2     = mean_absolute_error(to_predict['price'], tier2_pred)
mape_t2    = (abs(to_predict['price'] - tier2_pred) / to_predict['price']).mean() * 100

print(f"{'Tier 2 (last price + calib)':<35} Rp{mae_t2:>13,.0f} {mape_t2:>7.2f}%")

Tier 2 (last price + calib)         Rp       18,005    0.06%


#### Tier 1 vs Tier 2 Comparison

In [7]:
print(f"{'Model':<35} {'MAE':>14} {'MAPE':>8}")
print(f"{'Tier 1 + anchor calibration':<35} Rp{mae_t1:>13,.0f} {mape_t1:>7.2f}%")
print(f"{'Tier 2 (last price + calib)':<35} Rp{mae_t2:>13,.0f} {mape_t2:>7.2f}%")

Model                                          MAE     MAPE
Tier 1 + anchor calibration         Rp       47,124    0.20%
Tier 2 (last price + calib)         Rp       18,005    0.06%


Tier 2 wins because product prices are highly stable from day to day. Tier 1 (RandomForest) is more useful for new products that do not have a price history. Tier 2 cannot predict products without model_last_price.

Conclusion:
- Tier 2 for products with sufficient history (≥ 3 observations)
- Tier 1 as a fallback for new products (cold start)

In [8]:
import pickle

# Save model for prediction notebook
with open('model.pkl', 'wb') as f:
    pickle.dump({'model': model, 'FEATURE_COLS': FEATURE_COLS,
                 'global_median': global_median}, f)
print('✓ Saved: model.pkl')

✓ Saved: model.pkl
